## Sense-check salinity

Makes a few key plots of the final salinity files to check for possible bugs.  Note that there is an optional visualization included in `Step3-QDMsalinity` that will compare salinity in the reanalysis product, the raw CMIP model, and the bias-corrected dataset.

17 Apr 2026 | EHU -- based on oceanTF notebook by DS

#### Required files/inputs
- CMIP monthly bias-corrected ocean salinity. These examples use batched output of Step4 (e.g. *so_CESM2-WACCM-historical-1850_2014.nc*), but you may adapt the read-in to use annual files (e.g. *so_CESM2-WACCM_historical_1850.nc*)

#### Outputs
- Maps of salinity and salinity change over 20-yr time periods
- Time series of salinity near 4 large glaciers, showing both annual and sub-annual variability

In [ ]:
# imports
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# inputs
cmip_model = 'CESM2-WACCM' # cmip model
scenario = 'ssp585' # scenario

# directories
hist_dir = '/Users/eultee/Desktop/' + cmip_model + '/historical/so/' # historical files
proj_dir = '/Users/eultee/Desktop/' + cmip_model + '/' + scenario + '/so/' # projection files
plot_dir = '/Users/eultee/Desktop/' + cmip_model + '/plots/' # directory to save plots

# Glacier time series positions to extract (meters)
xpos = [-197975,  313225,   -273575,  459475]
ypos = [-2271580, -2578020, -938675, -1039020]
namepos = ['Jakobshavn', 'Helheim', 'Petermann', '79N']
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

### Make some map plots to check spatial patterns, general magnitude and differences.

Here's an example of what we'd expect to see. The figure contains maps of 20-yr mean salinity for three time periods (top row) and their differences relative to 1985–2014 (bottom row) for CESM2-WACCM under ssp585. Panel d shows the locations of the 4 time series in the next two plots.

<img src="./plots_CESM2-WACCM_ssp585_so_map.png" width="800">

Looking for
- Correct x and y axes (units are km).
- Sensible magnitudes for salinity. That is, probably 30-36% in 1985-2014 (in fact all climate models should look similar to panel a of this plot if the bias correction has worked well)
- Spatial patterns make sense: NaN values (white) over above sea-level pixels, non-NaN values elsewhere, small-scale variability linked to bathymetry (troughs, continental shelf edge).
- Changes in the future time periods that are appropriate order of magnitude (roughly 3-6, or 10-20% of the baseline salinity value in the example shown here)

In [ ]:
# map plots of salinity and salinity change
## these will be using batched files (1850-2014, 2015-2100, 2101-2299)
## see Donald's Step5-Sense-Checking in the TF folder for an example with annual files
import cftime

# periods
start_yrs = [1985, 2081, 2281]
end_yrs   = [2014, 2100, 2300]
means = [] 

for i, (start, end) in enumerate(zip(start_yrs, end_yrs)):
    if end <=2014:
        file = hist_dir + r'so_QDMCorrected-ISMIP_Grid-' + cmip_model + '-historical-1850_2014.nc'
    elif end <=2100:
        file = proj_dir + r'so_QDMCorrected-ISMIP_Grid-' + cmip_model + '-ssp585-2015_2100.nc'
    elif end <= 2300:
        file = proj_dir + r'so_QDMCorrected-ISMIP_Grid-' + cmip_model + '-ssp585-2101_2299.nc'
    else:
        print('End date out of bounds, no file found.  Please correct in end_yrs.')
    ds = xr.open_dataset(file)

    x = ds["x"]/1e3 # in km
    y = ds["y"]/1e3 # in km

    start_date='{}-01-01'.format(start) ## date string
    end_date = '{}-12-31'.format(end)
    cf_start = cftime.DatetimeNoLeap(*map(int, start_date.split('-'))) ## allow slicing
    cf_end = cftime.DatetimeNoLeap(*map(int, end_date.split('-')))
    
    SO_mean = ds['so'].sel(time=slice(cf_start,cf_end)).resample({'time': 'A'}).mean().mean(dim='time')
    means.append(SO_mean)

max_so = np.nanmax(means)
min_so_toshow = max_so - 10 ## ersatz lower limit for colorbars

# --- plotting ---
fig, axes = plt.subplots(2, 3, figsize=(12, 9))

# row 1: means
for i in range(3):
    pcm = axes[0, i].pcolormesh(x, y, means[i], vmin=min_so_toshow, vmax=max_so, shading="auto", cmap="viridis")
    axes[0, i].set_title(f"{start_yrs[i]}-{end_yrs[i]} mean salinity")
    axes[0, i].set_aspect("equal")
    fig.colorbar(pcm, ax=axes[0, i])

# row 2: differences relative to first period
for i in range(3):
    diff = means[i] - means[0]
    colmax = float(np.max(np.abs(diff)))
    if colmax==0:
        colmax=1
    pcm = axes[1, i].pcolormesh(x, y, diff, vmin=-colmax, vmax=colmax, shading="auto", cmap="RdBu_r")
    if i==0:
        for xnow, ynow, c, name in zip(np.array(xpos), np.array(ypos), colors, namepos):
            axes[1, i].plot(xnow/1e3, ynow/1e3, 'o', color=c, markersize=10, label=name)
        axes[1, i].legend()
        axes[1,i].set_title(f"Glacier site locations")
    else:
        axes[1, i].set_title(f"{start_yrs[i]}-{end_yrs[i]} vs {start_yrs[0]}-{end_yrs[0]} salinity")
        fig.colorbar(pcm, ax=axes[1, i])
    axes[1, i].set_aspect("equal")

fig.suptitle(cmip_model+'_'+scenario, fontsize=16)
plt.tight_layout()
plt.savefig(plot_dir+'plots_'+cmip_model+'_'+scenario+'_so_map.png', dpi=300, bbox_inches="tight")
plt.show()

### Time series plots to check general magnitude and trends

Here's an example of what we'd expect to see. The figure contains time series of  salinity, extracted from points just in front of the present calving fronts of 4 major glaciers, for CESM2-WACCM in a ssp585 scenario. We separately plot 1850-2014 and 2015-2300 since the scales can be very different.


<img src="./plots_CESM2-WACCM_ssp585_so_timeseries.png" width="800">

Looking for
- Sensible magnitudes for salinity. That is, values likely in the mid-30s for 1985-2014, and agreeing with the order of magnitude shown in the map plots above
- Seasonal variability throughout (due to input of runoff?) that is small compared with the mean value
- Temporal pattern consistent with climate scenario -- we are not familiar with the expected changes to salinity, but we would expect some slight differences across scenario.  In the ssp585 example shown, we see a decrease in salinity at all four sites through the 21st century, followed in the 22nd-23rd centuries by increases whose magnitude depends on location.

In [ ]:
# time series plots at glacier sites
## these will be using batched files (1850-2014, 2015-2100, 2101-2299)
## see Donald's Step5-Sense-Checking in the TF folder for an example loading in annual files

ds_hist = xr.open_dataset(hist_dir + r'so_QDMCorrected-ISMIP_Grid-{}-historical-1850_2014.nc'.format(cmip_model))
## open both projection periods in a single dataset
ds_proj = xr.open_mfdataset(proj_dir + r'so_QDMCorrected-ISMIP_Grid-{}-{}-*.nc'.format(cmip_model, scenario))


# annual mean plots
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

for i in range(len(xpos)):
    ## historical
    so_toplot_h = ds_hist.so.sel(x=xpos[i], y=ypos[i], method='nearest')
    so_toplot_h.plot(ax=axes[0], label=namepos[i], linewidth=1.5)

    ## projection
    so_toplot_p = ds_proj.so.sel(x=xpos[i], y=ypos[i], method='nearest')
    so_toplot_p.plot(ax=axes[1], label=namepos[i], linewidth=1.5)


for ax in axes:
    ax.set(xlabel="Year", ylabel='salinity [%]')
    # ax.set_ylabel("degC")
    ax.grid(True)

axes[0].set_title('Salinity, historical')
axes[1].set_title('Salinity, projection')
axes[1].legend(loc='upper left')

fig.suptitle(cmip_model+'_'+scenario, fontsize=16)
plt.tight_layout()
plt.savefig(plot_dir+'plots_'+cmip_model+'_'+scenario+'_so_timeseries.png', dpi=300, bbox_inches="tight")
plt.show()